# convert

Turns Hive someone else wrote, one query or a batch of scripts, into your own code: each table the scripts write becomes a Saved table in your Dev Database, with its Table folder, and what only you can decide of it, a Decision, answered in the decisions cell.

## Terms

- **Step**: each numbered heading below, with the code cell under it. Its first line says its kind: it changes only files, or only prints; it is a call to the warehouse, yours to run; or it is a decision, yours to make.
- **Names cell**: the code cell after the one that finds the Project. It holds this Workflow's inputs, the start materials you fill in once, each with what to write in its comment; each step's call reads them.
- **Decisions cell**: `DECISIONS`, the code cell after the names cell, where you answer each open Decision a step waits on, by the table's name.
- **Hive**: the SQL your warehouse runs.
- **Batch**: the scripts converted together, under one name.
- **Saved table**: a table in the warehouse that only your own code writes, its files in a folder of its own.
- **Dev Database**: the database where you create and write tables of your own, each built, written and compared there first: `DEV` in admin/databases.py.
- **Table folder**: the folder in tables/ that holds everything about one Saved table: its Table notebook, its past lineage, and its Hive.
- **Decision**: one thing about one table that only you decide, such as its location or its Key; the table's files say whether it is answered yet.
- **Table reference**: a script in references/ describing one table: its columns and their types, its Partitions and its Key.
- **Reads itself**: a table whose own script reads it, such as a running total, seeded once from its original before its first write.
- **Original**: the table a converted script writes, in the warehouse already, which its Saved table copies; for a query alone, such as one Copilot wrote from a workbook, the rows the query gives.
- **Session setting**: a `SET` line sent before a Statement, which it needs to run.
- **Partition**: the rows of a table that share one value of each column it is partitioned by.
- **Table notebook**: the notebook in a Table folder whose cells create, preview and write its Saved table; its logic cell is the one whose first line is `# logic`.
- **Key**: the columns whose values name exactly one row of a table.
- **Statement**: a query your code builds, which becomes one Hive string.

## Before you start

Fill in the names cell below and save the notebook: each call reads it, and so does `project.check()`, which names the next step.

## Decision rules

- If you want the steps run for you, as most do: fill in the names cell and save the notebook, run "Run up to your Decisions", paste what it prints into the decisions cell under it, fill it in and save, then run "Run the rest". Each runs every step left in turn, up to the first only you can do, and says where it stopped.
- If you run the steps yourself, run each step's cell under "Step by step", in turn, in order.
- If you want only the steps that change files run for you, run `project.do_next()`, the last cell. It runs them up to the first it can't.

## Self-check

Once the last step is done, each of these holds:

- Each table the scripts write has its Table folder in tables/ and its Table reference in references/.
- Each table's `conversion/checklist.md` says None under Open Decisions.
- admin/sent.csv records each table created and written, and each that reads itself seeded.
- Each table's `conversion/compare.md` says it passed, and the batch's folder is in admin/workflows/convert/finished/.
- `project.check()` names no next step of this batch, and names it under Done.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[2]))  # the Project: three up

from sqlglot_composer import admin, hive_settings, project, run_execute, run_query, show_hive
from warehouse import connect, query, execute

# Your connection to the warehouse. Each notebook runs on its own, so each connects here, at
# its top, before it calls query or execute: write yours in the line below and delete its #,
# such as connect(your_library.connect(...)). A connection kept in a file outside the Project,
# such as team_connection.py, is imported by these lines, written in that line's place:
#     sys.path.append("C:/work/connections")  # the folder team_connection.py is in
#     from team_connection import conn
#     connect(conn)
from sqlglot_composer import example_database
sys.path.append('C:\\Users\\myfir\\AppData\\Local\\Temp\\usability-loop\\runs\\2026-11-09-W8-2')  # the folder team_connection.py is in
from team_connection import conn
connect(conn)

In [ ]:
NAME = ''  # the batch's name, plain lowercase, such as "weekly" (or, for the run cells alone, a list of batches converted already); unused when EACH_SCRIPT_ITS_OWN_BATCH is True, each batch named for its file
FILES = []  # the scripts' files, such as ["daily.hql"], each beside this notebook or a path anywhere, such as "C:/work/daily.hql", read where it is: left [], every .hql file beside this notebook, or HIVE when it is given
HIVE = None  # the Hive as text, such as """SELECT ...""", or None to read FILES
EACH_SCRIPT_ITS_OWN_BATCH = False  # True makes each file of FILES a batch of its own, named for its file without .hql, such as "daily" for daily.hql, for simple scripts that read none of each other's tables: the run cells run each in turn, and one decisions cell answers them all
DATABASE = 'default'  # the database of a table the Hive names without one
RENAME = {}  # a name of your own for a table's copy in dev, by the table's full name, such as {"mart.team_fails": "team_fails"} for mart_dev.team_fails: left out, a copy is <table>_<its database>, such as mart_dev.team_fails_mart
STAGING = {}  # each staging table convert names, kept or folded, such as {"ops_reports.failed": "keep"}: left out, a temporary or DROPped one is folded, any other kept
PARTITIONS = []  # the Partitions for step 7 to compare besides the latest, each a dict, such as [{"dt": "2026-09-23"}], each value with no quotes of its own: [] compares only the latest Partition your dev table and its original both hold

## Run up to your Decisions

This cell and "Run the rest" are one call, `project.run_workflow`, run once before you answer and once after. Run the cell below: it runs every step left, up to the first only you can do, and says where it stopped. At a Decision, it prints a whole DECISIONS: paste it into the decisions cell under it, fill in each <FILL IN>, and save the notebook. With nothing waiting on you, it runs every step to the end, its calls to the warehouse too: give it dry_run=True first to see what it would send.

In [ ]:
# Run up to your Decisions: every step left, up to the first only you can do, such as a
# Decision, whose DECISIONS it prints to paste into the cell under it.
# Add dry_run=True to see the Hive it would send, sending nothing through your
# execute: help(project.run_workflow) says the steps it still runs:
project.run_workflow("convert", query=query, execute=execute)

In [ ]:
DECISIONS = {}  # your answer to each open Decision, by the table's name as
# project.check() writes it, such as
# {"mart_dev.weekly_ops": {"key": ["job_id"], "types": {"fails": "bigint"},
#                          "tags": {"PERIOD_END": "'2026-10-31'"}}}
# "key": None says no columns name one row. project.check() names each one open, and
# prints a whole DECISIONS to paste over this line: fill in each <FILL IN>. A Key
# convert or new_table proposed, marked # proposed, is kept unless you answer "key"
# here; one add_references proposed stays open until you accept it.
# "key": "proposed" accepts either as if typed. "all_tables" answers for every table
# of this notebook, and a table's own answer overwrites it, kind by kind and tag by
# tag:
# {"all_tables": {"key": "proposed",
#                 "tags": {"PERIOD_END": "'2026-10-31'"}}}
# accepts each table's proposed Key and gives each table holding PERIOD_END that
# value. A table's location is placed under "location_root" in
# admin/databases.py; answer "location" only for one it can't place.
# A tag may be answered with a Run value's function of your Project's, imported above
# DECISIONS, such as "PERIOD_END": period_end. For a table's latest Partition, write
# project.new_run_value("period_end", latest_partition="ops.job_runs.dt") at the top
# of this cell, the import it prints under it, and run the cell once: it writes the
# function into building_blocks/run_values.py. And "seed", such as
# {"dt": {"from": "2026-01-01"}}, limits the Partitions a table that reads itself
# copies: how-to 09 shows both.

## Run the rest

Run the cell below once the decisions cell above is filled in and saved: the same call again, it goes on from where the cell before stopped, and it too stops at the first step only you can do: a refusal, a compare that failed, or a Decision left open, each named. Mend what it names, such as an answer in the decisions cell, save, and run this cell again.

In [ ]:
# Run the rest: the same call, run again once you have answered: it goes
# on from where it stopped, and stops again at a refusal, a compare that
# failed or a Decision left open.
# Add dry_run=True to see the Hive it would send, sending nothing through your
# execute: help(project.run_workflow) says the steps it still runs:
project.run_workflow("convert", query=query, execute=execute)

# Step by step

Each step's own cell, to run or run again one step on purpose; the cells above run them all for you.

## 1. Convert the scripts: each table they write a Saved table in dev

It is a call to the warehouse, yours to run.

- **Goal:** a Table folder and a Table reference for each table the scripts write, each a Saved table in your Dev Database.
- **Action:** Run the cell below. It reads DESCRIBE through your query, and writes only files.
- **Done when** admin/workflows/convert/ holds the batch's folder, under its name, with its batch.md, which convert writes last; then step 2.

In [ ]:
project.convert(NAME, HIVE, files=FILES, query=query, database=DATABASE,
                rename=RENAME, staging=STAGING)

## 2. Answer each table's open Decisions in the decisions cell

It is a decision, yours to make.

- **Goal:** an answer to each open Decision of the batch's tables, each Script tag's value among them.
- **Action:** Run the cell below, `project.check()`: it names each open Decision and prints a whole `DECISIONS` to paste over the decisions cell's; then write each answer in the decisions cell, `DECISIONS`, by the table's name as `project.check()` writes it, each `<FILL IN>` given its answer, then save the notebook. Each table's `conversion/checklist.md`, in its Table folder, lists its open Decisions. A Script tag, a `<PARM="NAME"/>` the program that ran the scripts, such as a scheduler, replaced with a value before Hive read it, is answered under `"tags"` with that value as Hive. A tag standing alone, as in `dt <= <PARM="PERIOD_END"/>`, is answered with the value's own quotes: `{"tags": {"PERIOD_END": "'2026-10-31'"}}`. A tag already inside quotes, as in `'<PARM="SKIP_TEAM"/>'`, is answered without them: `{"tags": {"SKIP_TEAM": "nobody"}}`. To change one, answer it again and run steps 3, 6 and 7 again: the table is made already, so 4 and 5 stay done.
- **Done when** the decisions cell answers each Decision `project.check()` names for this batch; then step 3.

In [ ]:
project.check()

## 3. Write each answer into its table's Table reference

It changes only files, so project.do_next() runs it for you.

- **Goal:** each answer written into its table's Table reference.
- **Action:** Run the cell below.
- **Done when** each table's `conversion/checklist.md` says None under Open Decisions; then step 4.

In [ ]:
project.write_decisions(DECISIONS, tables=project.batch_tables(NAME))

## 4. Create each table in dev

It is a call to the warehouse, yours to run.

- **Goal:** each table of the batch made in your Dev Database.
- **Action:** Run the cell below. It sends each table's `create()` through your execute.
- **Done when** admin/sent.csv records a create of each table; then step 5.

In [ ]:
for table in project.batch_tables(NAME):
    run_execute(project.load(table).create(), execute)

## 5. Seed each table that reads itself, once, from its original

It is a call to the warehouse, yours to run.

- **Goal:** each table that reads itself holding its original's rows, once.
- **Action:** Run the cell below. It sends the Session setting their Partitions need, then each such table's `seed()`, through your execute. If no table of the batch reads itself, then this step is done once step 4 is, and the cell sends nothing and prints that it seeded nothing.
- **Done when** admin/sent.csv records the seed of each table that reads itself; then step 6.

In [ ]:
to_seed = project.batch_tables(NAME, seeded=True)
if not to_seed:
    print("Seeded nothing: no table of the batch reads itself.")
else:
    run_execute(hive_settings("dynamic_partitions"), execute)
    for table in to_seed:
        run_execute(project.load(table).seed(), execute)

## 6. Write each table in dev

It is a call to the warehouse, yours to run.

- **Goal:** each table's rows written in your Dev Database.
- **Action:** Run the cell below. `project.run_tables` sends each table's `write()` through your execute.
- **Done when** admin/sent.csv records each table written; then step 7.

In [ ]:
project.run_tables(tables=project.batch_tables(NAME), execute=execute)

## 7. Compare each table with its original

It is a call to the warehouse, yours to run.

- **Goal:** each table's rows checked against its original's: a partitioned table's on the latest Partition both hold, and on each other one PARTITIONS lists.
- **Action:** Run the cell below. `project.compare_batch` reads both through your query and writes each table's result in its `conversion/compare.md`. A partitioned table is compared on the latest Partition both tables hold, which its line in the report names, such as `on dt = '2026-09-24'`. If you want other Partitions compared too, each one your dev table holds as well as its original, then list them in `PARTITIONS` in the names cell, such as `[{"dt": "2026-09-23"}]`, each value with no quotes of its own, run the names cell, and run this step's cell again: each table partitioned by dt is compared on its latest and on each listed, the others on their latest, each result its own section, kept beside the ones before; a result that failed is replaced when that Partition is compared again. If the batch is finished already, then the same: list them in PARTITIONS and run this step's cell, or "Run the rest", which compares a finished batch again whenever PARTITIONS lists a Partition, and says the batch was finished.
- **Done when** every result in each table's `conversion/compare.md` says Passed, and the batch's folder has moved to admin/workflows/convert/finished/; a table with a result that failed is one promote refuses; the Workflow is done: go through the Self-check at the top.

In [ ]:
project.compare_batch(NAME, query=query, partitions=PARTITIONS)

In [ ]:
project.do_next()